# 02 — Data dictionary (Gastat and Sama)

Describes **every column of the outside data** in the Drive folder *Darb DataSets*: **Gastat** (3 files) and
**Sama** (2 files), and the **features** that can be made from them.
MC and Monshaat are not listed here (team decision).

**Phase 1: facts, measured by code.** For each column: its original name, type, missing share, number of different
values, min/max, format pattern and most common values.
**Phase 2: written details.** Meaning (English + Arabic), unit, quality issues, role, join key and leakage risk,
and the planned features.

**Rules for this notebook**
- The raw files are never changed. Results are saved as new files in the data folder, never in Git.
- Clear all outputs before committing.

### Cell 1 — Setup
Finds the data folder. On Colab the files are downloaded into `/content/darb-data` (temporary copies, gone when
the session ends). On a laptop it reads a local copy of the Drive folder.
`%pip` is used instead of `!pip` because `!pip` can hang in the VS Code Colab extension.

In [ ]:
import os
import gc
import re
from pathlib import Path

import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — The Gastat and Sama files
The 5 files, with the folder they are in and their Google Drive ID. Each file is downloaded once into
`DATA_DIR/<folder>/`, the same layout as `01_preprocessing`, so both notebooks find the same files.

In [ ]:
# (folder, file name, Google Drive ID)
FILES = [
    # Gastat (General Authority for Statistics)
    ('Gastat', 'gastat_gdp_2026-10-21T09_14_09.657Z.csv',                              '1fRoZuqgt1ltaQcXfSzbI2rk9mjK1pRH8'),
    ('Gastat', 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv',  '11WopaP1bfkU4bmeugGygvgxjorm56iE-'),
    ('Gastat', 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx', '1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ'),
    # Sama (Saudi Central Bank)
    ('Sama', 'point-of-sale-transactions-by-sector-and-city.xlsx',  '1iMoJekCkgIWCva3-H1RPxpGBJQ64oNz3'),
    ('Sama', 'points-of-sale-transactions-and-sales-by-sector.xlsx', '1ZZY1XCnTcFUjegFPrGvcWzIq1yW2OcQj'),
]
FOLDERS = ['Gastat', 'Sama']

def path_of(folder, name):
    return DATA_DIR / folder / name

if IN_COLAB:
    import gdown
    for folder, name, drive_id in FILES:
        path = path_of(folder, name)
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            print('downloading', folder, name)
            gdown.download(id=drive_id, output=str(path), quiet=True)

files = pd.DataFrame(FILES, columns=['folder', 'file', 'drive_id']).drop(columns='drive_id')
files['MB'] = [round(path_of(f, n).stat().st_size / 1e6, 1) if path_of(f, n).exists() else None
               for f, n in zip(files['folder'], files['file'])]
print(files.to_string())
print()
print(f'{files["MB"].notna().sum()} of {len(files)} files found | missing:',
      files.loc[files['MB'].isna(), 'file'].tolist() or 'none')

### Cell 3 — How each table starts
Statistics files often start with a title, notes or empty rows, so the code guesses the **header row**: the first
row that is (almost) as full as the fullest row near the top. This cell prints the first rows of every sheet with
the guessed header row, so we can check it.

**If a guess is wrong**, add the right row to `HEADER_ROW` (row numbers as printed on the left) and run this cell
again. These files hold only totals, not businesses, so showing their rows is safe.

In [ ]:
# Fixes for wrong header guesses: {(file name, sheet name): header row}
HEADER_ROW = {}

def read_raw(path):
    """Every sheet of a file as text, without a header: {sheet name: table}."""
    if path.suffix.lower() == '.csv':
        for encoding in ['utf-8-sig', 'cp1256']:
            try:
                return {'(csv)': pd.read_csv(path, header=None, dtype=str, encoding=encoding)}
            except UnicodeDecodeError:
                continue
        raise ValueError(f'Could not read {path.name}')
    return pd.read_excel(path, engine='calamine', header=None, dtype=str, sheet_name=None)

def find_header(raw):
    """First row near the top that is at least 80% as full as the fullest row there (titles have 1 cell)."""
    top = raw.head(25).notna().sum(axis=1)
    return int(top.index[top >= 0.8 * top.max()][0]) if len(top) and top.max() > 0 else 0

def with_header(raw, header_row):
    """Use one row as column names; the rows below it are the data."""
    names, seen = [], {}
    for position, value in enumerate(raw.iloc[header_row], 1):
        value = '' if pd.isna(value) else re.sub(r'\s+', ' ', str(value)).strip()
        value = value or f'(no name, column {position})'
        seen[value] = seen.get(value, -1) + 1
        names.append(value if seen[value] == 0 else f'{value}.{seen[value]}')
    table = raw.iloc[header_row + 1:].copy()
    table.columns = names
    table = table.dropna(how='all')
    unnamed_and_empty = [c for c in table.columns if c.startswith('(no name') and table[c].isna().all()]
    return table.drop(columns=unnamed_and_empty)

with pd.option_context('display.max_columns', 40, 'display.width', 250, 'display.max_colwidth', 30):
    for folder, name, _ in FILES:
        for sheet, raw in read_raw(path_of(folder, name)).items():
            header = HEADER_ROW.get((name, sheet), find_header(raw))
            print('=' * 100)
            print(f'{folder} | {name} | sheet: {sheet} | {raw.shape[0]:,} rows x {raw.shape[1]} columns | header row guess: {header}')
            print(raw.head(header + 6).fillna('').to_string())
            print()

### Cell 4 — Profile every column of every file
For each column of each sheet of each file, the code measures:

| Field | Meaning |
|---|---|
| `kind` | number, date, text, or empty (what at least 95% of the filled values look like) |
| `missing %` | share of empty cells |
| `unique` | number of different values |
| `min` / `max` | for numbers and dates |
| `top values` | the 10 most common values with their share |
| `patterns` | the 3 most common shapes: `9×4` = 4 digits, `ع` = an Arabic word, `a` = a Latin word |

**Why:** these facts come from the data, not from guessing, and the cell can be run again whenever the data changes.

In [ ]:
PATTERN_SAMPLE = 200_000

def shape(values):
    """Replace digits with 9×n, Arabic words with ع and Latin words with a."""
    out = values.str.replace(r'\d+', lambda m: '9' if len(m.group()) == 1 else f'9×{len(m.group())}', regex=True)
    out = out.str.replace(r'[ء-ْٱ-ۓ]+', 'ع', regex=True)
    out = out.str.replace(r'[A-Za-z]+', 'a', regex=True)
    return out.str.replace(r'\s+', ' ', regex=True)

def share_list(counts, total, limit):
    return ' | '.join(f'{value} ({100 * count / total:.1f}%)' for value, count in counts.head(limit).items())

def profile_column(column):
    values = column.astype('string').str.strip()
    values = values.mask(values == '')
    filled = values.dropna()
    row = {'rows': len(values),
           'missing %': round(100 * (1 - len(filled) / len(values)), 1) if len(values) else None,
           'unique': int(filled.nunique())}
    if filled.empty:
        return {**row, 'kind': 'empty'}
    sample = filled.sample(min(len(filled), PATTERN_SAMPLE), random_state=0)
    numbers = pd.to_numeric(filled.str.replace(',', '', regex=False), errors='coerce')
    if numbers.notna().mean() >= 0.95:
        row['kind'] = 'number'
        row['min'], row['max'] = numbers.min(), numbers.max()
    elif sample.str.match(r'^\d{4}-\d{2}-\d{2}').mean() >= 0.95:
        row['kind'] = 'date'
        row['min'], row['max'] = filled.str[:10].min(), filled.str[:10].max()
    else:
        row['kind'] = 'text'
    if row['kind'] == 'text' or row['unique'] <= 20:
        row['top values'] = share_list(filled.value_counts(), len(filled), 10)
    row['patterns'] = share_list(shape(sample).value_counts(), len(sample), 3)
    return row

inventory, profile = [], []
for folder, name, _ in FILES:
    for sheet, raw in read_raw(path_of(folder, name)).items():
        header = HEADER_ROW.get((name, sheet), find_header(raw))
        table = with_header(raw, header)
        table.columns = [str(c) for c in table.columns]
        inventory.append({'folder': folder, 'file': name, 'sheet': sheet,
                          'header row': header, 'rows': len(table), 'columns': table.shape[1]})
        for position, column in enumerate(table.columns, 1):
            profile.append({'folder': folder, 'file': name, 'sheet': sheet, 'position': position,
                            'column': column, **profile_column(table[column])})
        print(f'{folder:<7} {name[:70]:<70} | {sheet[:25]:<25} | {len(table):>9,} rows | {table.shape[1]:>3} columns')
    gc.collect()

inventory = pd.DataFrame(inventory)
profile = pd.DataFrame(profile)
ORDER = ['folder', 'file', 'sheet', 'position', 'column', 'kind', 'rows',
         'missing %', 'unique', 'min', 'max', 'top values', 'patterns']
profile = profile.reindex(columns=ORDER)
print()
print(f'{len(inventory)} sheets in {inventory["file"].nunique()} files | {len(profile):,} columns profiled')

### Cell 5 — The profile, file by file
Long text is cut to 120 characters here only; the full text is in the saved Excel file (Cell 7).

In [ ]:
def show(table):
    cut = table.copy()
    for column in ['top values', 'patterns']:
        cut[column] = cut[column].fillna('').str.slice(0, 120)
    with pd.option_context('display.max_columns', 30, 'display.width', 300, 'display.max_colwidth', 120):
        print(cut.drop(columns=['folder', 'rows']).fillna('').to_string(index=False))

for folder in FOLDERS:
    print('=' * 100)
    print(folder)
    show(profile[profile['folder'] == folder])
    print()

### Cell 6 — Which columns appear in which file
One row per column, ✅ for each file and sheet of the folder it appears in.

In [ ]:
by_file = {}
for folder in FOLDERS:
    part = profile[profile['folder'] == folder]
    by_file[folder] = (part.assign(seen='✅', where=part['file'] + ' | ' + part['sheet'])
                       .pivot_table(index='column', columns='where', values='seen', aggfunc='first').fillna('—'))
    print(f'{folder} columns by file and sheet:')
    with pd.option_context('display.width', 300, 'display.max_colwidth', 60):
        print(by_file[folder].to_string())
    print()

### Cell 7 — Save the Phase 1 profile
Saves `data_dictionary_profile.xlsx` in the data folder (never in Git), with the sheets `inventory`, `columns`,
`gastat_by_file` and `sama_by_file`.

In [ ]:
OUT = DATA_DIR / 'data_dictionary_profile.xlsx'
with pd.ExcelWriter(OUT) as writer:
    inventory.to_excel(writer, sheet_name='inventory', index=False)
    profile.to_excel(writer, sheet_name='columns', index=False)
    for folder, table in by_file.items():
        table.to_excel(writer, sheet_name=f'{folder.lower()}_by_file')
print('Saved', OUT)

## Phase 2 — Written details

Phase 1 measured the facts. Phase 2 adds what code cannot know: what each column **means**, how it can be used,
and what can go wrong. Written from the Phase 1 results (Cells 3, 5 and 6).

**Fields**

| Field | Meaning |
|---|---|
| `meaning (English)` / `المعنى (عربي)` | what the column represents |
| `unit / format` | e.g. SAR, count, date |
| `quality issues` | problems found in Phase 1 |
| `role` | **helper** (used to build something else), **feature** (candidate model input), **reference** (lookup table) |
| `join key` | how the column attaches to one business |
| `leakage risk` | can the value carry information from *after* the business started? |

**The leakage rule used everywhere:** a feature may only use information available **before or at the business's
start quarter**. For Gastat and Sama this means the latest period that ends **before** the start quarter.

**The business's sector** (the join key of most features) is `keywords_then_rules_section` /
`keywords_then_rules_division` from `03_official_activity` (empty sector = `"unclassified"`).

### Cell 8 — Description of every column

In [ ]:
DESCRIPTIONS = []

def describe(folder, file, column, en, ar, unit, issues, role, join, leakage):
    DESCRIPTIONS.append({'folder': folder, 'file': file, 'column': column,
                         'meaning (English)': en, 'المعنى (عربي)': ar, 'unit / format': unit,
                         'quality issues': issues, 'role': role, 'join key': join, 'leakage risk': leakage})

# ---------------------------------------------------------------- Gastat: GDP
GDP = 'gastat_gdp_2026-10-21T09_14_09.657Z.csv'
describe('Gastat', GDP, 'Quarter ID', 'Quarter as a number (year + quarter).', 'رمز الربع (السنة + رقم الربع).',
    'YYYYQ, e.g. 20101', 'None', 'helper', 'Quarter (time join)', 'Use quarters before the start quarter only')
describe('Gastat', GDP, 'Quarter', 'Quarter label, 2010 Q1 to 2026 Q2 (66 quarters).', 'الربع، من الربع الأول 2010 إلى الربع الثاني 2026.',
    'YYYY-Qn', 'None', 'helper', 'Quarter (time join)', 'Use quarters before the start quarter only')
describe('Gastat', GDP, 'Economic Activity Section ID', 'Gastat\'s own number for the economic activity (1–12).', 'رقم النشاط الاقتصادي حسب الهيئة (1–12).',
    'number 1–12', 'Not the SSIC letters: 12 groups instead of 21 sections, some merged (e.g. "Electricity, Gas and Water").', 'helper', 'Needs a mapping table to the SSIC section', 'None')
describe('Gastat', GDP, 'Economic Activity Section', 'English name of the economic activity.', 'اسم النشاط الاقتصادي بالإنجليزية.',
    'English text', 'English only.', 'helper', 'Same as Section ID', 'None')
describe('Gastat', GDP, 'GDP', 'Gross domestic product of the activity in the quarter.', 'الناتج المحلي الإجمالي للنشاط في الربع.',
    'Not stated; the scale matches million SAR. Current or constant prices not stated (to confirm on the Gastat portal).', 'Unit and price basis unknown. Recent quarters may be revised later.',
    'feature', 'Sector (via mapping) + the quarter before the start', 'High for quarters at or after the start; use only earlier quarters (e.g. growth over the 4 quarters before the start)')

# ---------------------------------------------------------------- Gastat: SME expenses and revenues
SME = 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv'
describe('Gastat', SME, 'Economic Sectors ID', 'Economic sector as an SSIC section letter (17 sectors, B onwards).', 'القطاع الاقتصادي برمز باب التصنيف (17 قطاعاً).',
    'letter', 'Agriculture (A) and some sections are not included.', 'helper', 'Joins directly to the SSIC section', 'None')
describe('Gastat', SME, 'Economic Sectors', 'English name of the sector.', 'اسم القطاع بالإنجليزية.', 'English text', 'None', 'helper', 'Same as Sectors ID', 'None')
describe('Gastat', SME, 'Establishment Size ID', 'Size class number (1 micro, 2 small, 3 medium).', 'رقم فئة الحجم (1 متناهية الصغر، 2 صغيرة، 3 متوسطة).',
    'number 1–3', 'MC has no size column; most new businesses are micro.', 'helper', 'Size class (assume micro for new businesses)', 'None')
describe('Gastat', SME, 'Establishment Size', 'Size class name.', 'اسم فئة الحجم.', 'English text', 'None', 'helper', 'Same as Size ID', 'None')
describe('Gastat', SME, 'Flow ID', 'Number of the flow (1 expenses, 2 revenues).', 'رقم نوع التدفق (1 مصروفات، 2 إيرادات).', 'number 1–2', 'None', 'helper', 'Chooses expenses or revenues', 'None')
describe('Gastat', SME, 'Flow', 'Expenses or Revenues.', 'مصروفات أو إيرادات.', 'English text', 'None', 'helper', 'Same as Flow ID', 'None')
describe('Gastat', SME, 'Year', 'Year of the figures (2019–2024).', 'سنة البيانات (2019–2024).', 'YYYY', 'Yearly only.', 'helper', 'Year (time join)', 'Use the year before the start year')
describe('Gastat', SME, 'Value', 'Total operating expenses or revenues of SMEs in the sector, size class and year.', 'إجمالي المصروفات أو الإيرادات التشغيلية للمنشآت في القطاع والحجم والسنة.',
    'Not stated (likely thousand SAR; to confirm)', 'Unit unknown. 2023 values jump for some sectors (check).',
    'feature', 'Sector + size + the year before the start (e.g. revenue / expense ratio)', 'High for the start year or later; use the year before')

# ---------------------------------------------------------------- Gastat: SSIC
SSIC = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
for column, en, ar, unit in [
    ('رمز الباب', 'Section code: an Arabic letter, 21 sections mapped to A–U in their order (أ=A, ب=B … ش=T).', 'رمز الباب (حرف عربي).', '1 Arabic letter'),
    ('اسم الباب', 'Section name.', 'اسم الباب.', 'Arabic text'),
    ('رمز القسم', 'Division code.', 'رمز القسم.', '2 digits'),
    ('اسم القسم', 'Division name.', 'اسم القسم.', 'Arabic text'),
    ('رمز المجموعة', 'Group code.', 'رمز المجموعة.', '3 digits'),
    ('اسم المجموعة', 'Group name.', 'اسم المجموعة.', 'Arabic text'),
    ('رمز الفئة', 'Class code.', 'رمز الفئة.', '4 digits'),
    ('اسم الفئة', 'Class name.', 'اسم الفئة.', 'Arabic text'),
    ('رمز النشاط', 'Activity code (most detailed level).', 'رمز النشاط (أدق مستوى).', '6 digits'),
    ('مسمى النشاط', 'Activity name.', 'مسمى النشاط.', 'Arabic text'),
    ('شرح النشاط', 'Description of what the activity includes and excludes.', 'شرح ما يشمله النشاط وما لا يشمله.', 'Arabic text'),
]:
    describe('Gastat', SSIC, column, en, ar, unit,
             'Codes are stored as numbers in Excel, so leading zeros are lost unless read as text.' if 'رمز' in column and column != 'رمز الباب' else 'None',
             'reference', 'Sector / division of each business', 'None (classification, not data about businesses)')
describe('Gastat', SSIC, 'الجهة المشرفة', 'Authority that supervises the activity.', 'الجهة المشرفة على النشاط.', 'Arabic text', 'None',
    'reference', 'Via the division (possible feature: regulated activity)', 'None')
describe('Gastat', SSIC, 'الجهة المرخصة', 'Authority that licenses the activity.', 'الجهة المرخصة للنشاط.', 'Arabic text', 'None',
    'reference', 'Via the division (possible feature: needs a license outside MC)', 'None')

# ---------------------------------------------------------------- Sama: transactions by sector and city
SAMA_CITY = 'point-of-sale-transactions-by-sector-and-city.xlsx'
describe('Sama', SAMA_CITY, 'Starting date', 'First day of the week the figures cover.', 'تاريخ بداية الأسبوع.',
    'date, weekly, 2020-05-10 to 2025-07-06', 'Ends in July 2025.', 'helper', 'Date (time join)', 'Use weeks before the start quarter only')
describe('Sama', SAMA_CITY, 'Indicator', 'What the value measures: number of transactions (thousands), value of transactions (thousand SAR), and their % change.', 'المؤشر: عدد العمليات (بالآلاف)، قيمة العمليات (بآلاف الريالات)، ونسبة التغير.',
    'category, 4 values', 'Levels and % changes are mixed in one value column (the source of the negative values).', 'helper', 'Choose one indicator before use', 'None')
describe('Sama', SAMA_CITY, 'Sectors', 'Spending sector (e.g. clothing, hotels, food and beverage), 17 sectors + Total.', 'قطاع الإنفاق، 17 قطاعاً + الإجمالي.',
    'English text', 'Not SSIC: needs a Sama-sector → SSIC mapping. Sector rows exist only for the whole country (City = Total): there are no city × sector numbers. "Total" rows must not be added to the sectors.', 'helper', 'Sector (via mapping)', 'None')
describe('Sama', SAMA_CITY, 'City', 'City, in English capitals: 10 cities + OTHER (every other city) + Total.', 'المدينة بالإنجليزية: 10 مدن + "أخرى" + الإجمالي.',
    'English text', 'Only 10 named cities. City rows exist only for all sectors together (Sector = Total). "Total" rows must not be added to the cities.', 'helper', 'City of the business (English → Arabic mapping)', 'None')
describe('Sama', SAMA_CITY, 'value (Multiple units)', 'The value of the indicator for the week and the sector (whole country) or the city (all sectors).', 'قيمة المؤشر للأسبوع وللقطاع (على مستوى المملكة) أو للمدينة (كل القطاعات).',
    'depends on Indicator', 'Unit depends on the Indicator column.', 'feature', 'Sector OR city (not both) + a fixed year before the first cohort', 'High for weeks at or after the start; the features use a fixed year that ends before the first cohort')

# ---------------------------------------------------------------- Sama: transactions and sales by sector
SAMA_SECTOR = 'points-of-sale-transactions-and-sales-by-sector.xlsx'
describe('Sama', SAMA_SECTOR, 'Periodicity', 'Monthly, Quarterly or Annually.', 'دورية البيانات: شهري، ربعي، سنوي.',
    'category, 3 values', 'The three periodicities are mixed in one table: choose one, or the same spending is counted three times.', 'helper', 'Filter to one periodicity', 'None')
describe('Sama', SAMA_SECTOR, 'Year', 'Year (2016–2023).', 'السنة (2016–2023).', 'YYYY', 'Ends in 2023.', 'helper', 'Year (time join)', 'Low: all years are before the cohorts')
describe('Sama', SAMA_SECTOR, 'Month', 'Month for monthly rows.', 'الشهر للبيانات الشهرية.', 'JAN … DEC', 'Empty for quarterly and annual rows (29% missing).', 'helper', 'Month (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Quarter', 'Quarter for quarterly rows.', 'الربع للبيانات الربعية.', 'Q1 … Q4', 'Empty for monthly and annual rows (77% missing).', 'helper', 'Quarter (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Transactions / Sales', 'Actually the SECTOR (e.g. Hotels): the two column titles are swapped in the source file.', 'في الحقيقة هو القطاع: عنوانا العمودين مقلوبان في الملف.',
    'English text, 17 values', 'Header swapped with "Sector". Not SSIC: needs a mapping.', 'helper', 'Sector (via mapping)', 'None')
describe('Sama', SAMA_SECTOR, 'Sector', 'Actually the INDICATOR: Number of Transactions or Sales (titles swapped in the source file).', 'في الحقيقة هو المؤشر: عدد العمليات أو المبيعات.',
    'category, 2 values', 'Header swapped with "Transactions / Sales".', 'helper', 'Choose one indicator', 'None')
describe('Sama', SAMA_SECTOR, 'Value in different units', 'Value of the indicator.', 'قيمة المؤشر.',
    'Not stated (likely thousands of transactions / thousand SAR, as in the city file)', 'Unit depends on the indicator.', 'feature', 'Sector + year before the start (e.g. sales growth)', 'Low: ends in 2023, before every cohort (but old for 2025 cohorts)')
describe('Sama', SAMA_SECTOR, 'Date_Object', 'First day of the period as a date.', 'أول يوم في الفترة كتاريخ.', 'date', 'None', 'helper', 'Date (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Date', 'Period label (e.g. "Q2 2023", "MAR 2023", "2023").', 'وصف الفترة.', 'text', 'Duplicates Year/Month/Quarter.', 'helper', 'None (use Date_Object)', 'None')
describe('Sama', SAMA_SECTOR, 'Name', 'Sector and indicator joined ("Hotels : Sales").', 'القطاع والمؤشر معاً.', 'text', 'Duplicates two other columns.', 'helper', 'None', 'None')

descriptions = pd.DataFrame(DESCRIPTIONS)
print(f'{len(descriptions)} descriptions written')

# Check: every profiled column has a description, and every description matches a profiled column
profiled = set(zip(profile['folder'], profile['file'], profile['column']))
described = set(zip(descriptions['folder'], descriptions['file'], descriptions['column']))
print('Columns without a description:', sorted(profiled - described) or 'none')
print('Descriptions without a column:', sorted(described - profiled) or 'none')

### Cell 9 — Mapping tables (agreed by the team)
Gastat GDP and Sama use their own groups, not the SSIC sections of our businesses. These tables translate them, so
each business can be joined to the outside figures through its sector (and city).

**1. Gastat GDP activity (1–12) → SSIC sections.** One Gastat group can hold several sections, which then share the
group's figure (e.g. a shop (G) and a restaurant (I) both get group 6). J (information and communication) is in
group 7 ("…Communication"). Private schools and clinics (P, Q) are in group 9, not 10 (10 is the government itself).
Groups 11 (net taxes) and 12 (total GDP) are not sectors; 12 can be a fallback for unclassified businesses.

**2. Sama sector → SSIC section (and division).** A Sama sector is the type of shop where people pay by card.
- **G** has 7 Sama sectors that SSIC cannot tell apart (all division 47): a G business gets their **sum**.
- **I** is split by division: hotels (I 55) get `Hotels`, restaurants and cafés (I 56) get `Restaurants & Café`.
- **F** (construction) gets `Construction & Building Materials`, as a sign of building activity (team choice A).
- `Miscellaneous Goods and Services` and `Other(s)` are too mixed: not used. `Total` is not a sector.
- Sections with no Sama sector stay empty: A, B, C, K, L, M, N, O, S, T, U.

**3. Sama city (English) → MC city (Arabic).** `OTHER` covers every other city, so no business is left without a
city. The Arabic spelling must match MC's city column exactly: to check when the feature is built.

**Gastat SME** needs no table: it already uses SSIC letters (A, O, T, U are not in it).

The cell checks that every value in the files is in a table (or marked as not used), and the other way round.

In [ ]:
NOT_USED = ''   # mapped to nothing on purpose

GDP_MAP = pd.DataFrame([
    (1, 'Agriculture, Forestry & Fishing', 'A'),
    (2, 'Mining & Quarrying', 'B'),
    (3, 'Manufacturing', 'C'),
    (4, 'Electricity, Gas and Water', 'D'), (4, 'Electricity, Gas and Water', 'E'),
    (5, 'Construction', 'F'),
    (6, 'Wholesale & Retail Trade, Restaurants & hotels', 'G'), (6, 'Wholesale & Retail Trade, Restaurants & hotels', 'I'),
    (7, 'Transport, Storage & Communication', 'H'), (7, 'Transport, Storage & Communication', 'J'),
    (8, 'Finance, Insurance, Real Estate & Business Services', 'K'), (8, 'Finance, Insurance, Real Estate & Business Services', 'L'),
    (8, 'Finance, Insurance, Real Estate & Business Services', 'M'), (8, 'Finance, Insurance, Real Estate & Business Services', 'N'),
    (9, 'Community, Social & Personal Services', 'P'), (9, 'Community, Social & Personal Services', 'Q'),
    (9, 'Community, Social & Personal Services', 'R'), (9, 'Community, Social & Personal Services', 'S'),
    (9, 'Community, Social & Personal Services', 'T'),
    (10, 'Government Services', 'O'),
    (11, 'Net Taxes on Products', NOT_USED),
    (12, 'Gross Domestic Product', NOT_USED),
], columns=['gdp_id', 'gdp_activity', 'ssic_section'])

SAMA_SECTOR_MAP = pd.DataFrame([
    # Sama sector, SSIC section, SSIC division ('' = every division of the section)
    ('Beverage and Food', 'G', ''),
    ('Clothing and Footwear', 'G', ''),
    ('Electronic & Electric Devices', 'G', ''),
    ('Furniture', 'G', ''),
    ('Jewelry', 'G', ''),
    ('Gas Stations', 'G', ''),
    ('Construction & Building Materials', 'G', ''),
    ('Construction & Building Materials', 'F', ''),
    ('Hotels', 'I', '55'),
    ('Restaurants & Café', 'I', '56'),
    ('Education', 'P', ''),
    ('Health', 'Q', ''),
    ('Recreation and Culture', 'R', ''),
    ('Transportation', 'H', ''),
    ('Telecommunication', 'J', ''),
    ('Public Utilities', 'D', ''), ('Public Utilities', 'E', ''),
    ('Miscellaneous Goods and Services', NOT_USED, ''),
    ('Other', NOT_USED, ''),
    ('Others', NOT_USED, ''),
    ('Total', NOT_USED, ''),
], columns=['sama_sector', 'ssic_section', 'ssic_division'])

SAMA_CITY_MAP = pd.DataFrame([
    ('RIYADH', 'الرياض'),
    ('JEDDAH', 'جدة'),
    ('MAKKAH', 'مكة المكرمة'),
    ('MADINA', 'المدينة المنورة'),
    ('DAMMAM', 'الدمام'),
    ('KHOBAR', 'الخبر'),
    ('ABHA', 'أبها'),
    ('BURAIDAH', 'بريدة'),
    ('TABOUK', 'تبوك'),
    ('HAIL', 'حائل'),
    ('OTHER', '(every other city)'),
    ('Total', NOT_USED),
], columns=['sama_city', 'mc_city'])

def values_of(folder, file, column):
    """The different values of one column of a file (all sheets)."""
    out = set()
    for sheet, raw in read_raw(path_of(folder, file)).items():
        table = with_header(raw, HEADER_ROW.get((file, sheet), find_header(raw)))
        if column in table.columns:
            out |= set(table[column].dropna().astype(str).str.strip())
    return out

def check(label, in_file, in_table):
    print(f'{label}: {len(in_file)} values | not in the table: {sorted(in_file - in_table) or "none"} '
          f'| in the table but not in the file: {sorted(in_table - in_file) or "none"}')

check('Gastat GDP activities', values_of('Gastat', GDP, 'Economic Activity Section'), set(GDP_MAP['gdp_activity']))
sama_sectors = values_of('Sama', SAMA_CITY, 'Sectors') | values_of('Sama', SAMA_SECTOR, 'Transactions / Sales')
check('Sama sectors (both files)', sama_sectors, set(SAMA_SECTOR_MAP['sama_sector']))
check('Sama cities', values_of('Sama', SAMA_CITY, 'City'), set(SAMA_CITY_MAP['sama_city']))

LETTERS = list('ABCDEFGHIJKLMNOPQRSTU')
print()
print('SSIC sections without a GDP group:', [l for l in LETTERS if l not in set(GDP_MAP['ssic_section'])])
print('SSIC sections without a Sama sector (stay empty):', [l for l in LETTERS if l not in set(SAMA_SECTOR_MAP['ssic_section'])])

MAPPINGS = {'map_gdp_to_ssic': GDP_MAP, 'map_sama_sector_to_ssic': SAMA_SECTOR_MAP, 'map_sama_city_to_mc': SAMA_CITY_MAP}
for name, table in MAPPINGS.items():
    table.to_csv(DATA_DIR / f'{name}.csv', index=False, encoding='utf-8-sig')
print()
print('Saved', ', '.join(f'{name}.csv' for name in MAPPINGS), 'to', DATA_DIR)

### Cell 10 — Planned features (proposal for the team)
Features the model can use, built from the Gastat and Sama columns above. **Status:** `building` = being built in
`04_gastat_preprocessing` / `05_sama_preprocessing`; `do not use` = decided against, with the reason. Every one
follows the leakage rule (only information from before the start quarter).
`needs first` lists what must exist before the feature can be built.

In [ ]:
PLANNED = pd.DataFrame([
    # feature, how it is made, sources, join key, needs first, leakage note, status
    ('sector_gdp_growth', 'Growth of the sector\'s GDP over the 4 quarters before the start quarter.',
     'Gastat GDP', 'sector + quarter', 'Done: map_gdp_to_ssic (Cell 9)',
     'Safe only with quarters before the start', 'building (04_gastat_preprocessing)'),
    ('sme_revenue_expense_ratio', 'Revenues / expenses of micro SMEs in the sector, the year before the start.',
     'Gastat SME', 'sector + year', 'None (already SSIC letters); sectors not in the file stay empty',
     'Safe only with the year before the start', 'building (04_gastat_preprocessing)'),
    ('regulated_activity', 'Whether the division\'s licensing authority is outside the Ministry of Commerce.',
     'Gastat SSIC', 'division', 'The business\'s division (keywords_then_rules_division)', 'Safe', 'building (04_gastat_preprocessing)'),
    ('pos_sector_*', 'Point-of-sale spending in the business\'s SECTOR, whole country, over ONE FIXED YEAR (Oct 2023 – Sep 2024), the same for every business and for the website: yearly value, count, average ticket, share of all spending (built in 05_sama_preprocessing).',
     'Sama city file (sector rows)', 'sector (+ division for I)', 'Done: map_sama_sector_to_ssic (Cell 9)',
     'Safe: the fixed year ends before the first labelled cohort (2024 Q4)', 'building (05_sama_preprocessing)'),
    ('pos_city_*', 'Point-of-sale spending in the business\'s CITY, all sectors together, over the same fixed year: yearly value, count, average ticket, share of all spending. Cities outside the 10 use SAMA\'s OTHER row.',
     'Sama city file (city rows)', 'city of the first quarter', 'Done: map_sama_city_to_mc (Cell 9); city from 01 Cell 19; spelling check in 05 Cell D1',
     'Safe: the fixed year ends before the first labelled cohort (2024 Q4)', 'building (05_sama_preprocessing)'),
    ('NOT: pos_spending_city_sector', 'Spending in the city AND sector together.',
     'Sama city file', '—', '—',
     'Cannot be built: the file has no city × sector numbers (only sector rows for the whole country and city rows for all sectors)', 'do not use'),
    ('NOT: pos_sales_sector_trend', 'Yearly growth of point-of-sale sales in the sector.',
     'Sama sector file', '—', '—',
     'Not used: the file ends in 2023, and a growth number goes out of date (same reason as the fixed year)', 'do not use'),
], columns=['feature', 'how it is made', 'sources', 'join key', 'needs first', 'leakage', 'status'])

with pd.option_context('display.width', 250, 'display.max_colwidth', 70):
    print(PLANNED[['feature', 'sources', 'join key', 'needs first']].to_string(index=False))

### Cell 11 — The full data dictionary
Joins the descriptions (Cell 8) with the measured facts (Cell 4): one row per column of each file.

Saves `data_dictionary.xlsx` in the data folder (never in Git) with the sheets `dictionary`, `planned_features`,
the three mapping tables (`map_…`), `inventory` and `profile`. Cell 12 downloads it.

In [ ]:
FILE_PERIODS = {
    GDP: '2010Q1–2026Q2 (quarterly)',
    SME: '2019–2024 (yearly)',
    SSIC: 'version of 15 Sep 2026',
    SAMA_CITY: '2020-05-10 – 2025-07-06 (weekly)',
    SAMA_SECTOR: '2016–2023 (monthly, quarterly, yearly)',
}

def joined(values):
    return ' / '.join(dict.fromkeys(str(v) for v in values.dropna()))

def first_valid(values):
    values = values.dropna()
    return values.iloc[0] if len(values) else ''

facts = (profile.groupby(['folder', 'file', 'column'], sort=False)
         .agg(sheet=('sheet', joined), kind=('kind', first_valid), **{'missing %': ('missing %', 'max')},
              unique=('unique', 'max'), min=('min', first_valid), max=('max', first_valid),
              **{'top values': ('top values', first_valid), 'patterns': ('patterns', first_valid)})
         .reset_index())
facts['period'] = facts['file'].map(FILE_PERIODS)

DICT_ORDER = ['folder', 'file', 'period', 'sheet', 'column',
              'meaning (English)', 'المعنى (عربي)', 'unit / format', 'role', 'join key', 'leakage risk', 'quality issues',
              'kind', 'missing %', 'unique', 'min', 'max', 'top values', 'patterns']
dictionary = (facts.merge(descriptions, on=['folder', 'file', 'column'], how='inner')
              .reindex(columns=DICT_ORDER).reset_index(drop=True))

print(f'Data dictionary: {len(dictionary)} rows | by folder:', dictionary['folder'].value_counts().to_dict())
print()
print('Roles:')
print(dictionary['role'].value_counts().to_string())
print()
with pd.option_context('display.width', 250, 'display.max_colwidth', 45):
    print(dictionary[['folder', 'column', 'period', 'role', 'join key']].to_string(index=False))

OUT = DATA_DIR / 'data_dictionary.xlsx'
with pd.ExcelWriter(OUT) as writer:
    dictionary.to_excel(writer, sheet_name='dictionary', index=False)
    PLANNED.to_excel(writer, sheet_name='planned_features', index=False)
    for name, table in MAPPINGS.items():
        table.to_excel(writer, sheet_name=name, index=False)
    inventory.to_excel(writer, sheet_name='inventory', index=False)
    profile.to_excel(writer, sheet_name='profile', index=False)
print()
print('Saved', OUT)

### Cell 12 — Download the Excel files
The files are on the Colab server, not on your laptop, and are deleted when the session ends.
This cell shows a **download link** for each file (works inside VS Code). On colab.research.google.com it also
starts the download directly.

In [ ]:
import base64
from IPython.display import HTML, display

XLSX = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
for path in [DATA_DIR / 'data_dictionary.xlsx', DATA_DIR / 'data_dictionary_profile.xlsx']:
    if not path.exists():
        print('Not created yet:', path.name)
        continue
    data = base64.b64encode(path.read_bytes()).decode()
    display(HTML(f'<a download="{path.name}" href="data:{XLSX};base64,{data}">⬇ Download {path.name}</a> '
                 f'({path.stat().st_size / 1e3:.0f} KB)'))

if IN_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(DATA_DIR / 'data_dictionary.xlsx'))
    except Exception as error:
        print('Direct download is not available here; use the link above.', type(error).__name__)

### Notes — open questions for the team
- **Units** not stated in the files: Gastat GDP (likely million SAR; current or constant prices?), Gastat SME value,
  Sama values. To confirm on each source's website.
- **Mapping tables:** done and agreed (Cell 9). Still to check when the features are built: the Arabic spelling of
  the 10 Sama cities against MC's city column.
- **Periods:** Sama city data ends July 2025 and Sama sector data ends 2023, so the newest cohorts get older
  figures (or none). Gastat SME data ends 2024.